# Week 6 Day 5 – Optimized Fine-Tuning for Price Prediction

This notebook applies **optimization techniques** to the Day 5 fine-tuning pipeline to reduce **average absolute error** on the test set.

**Run from:** `week6` directory (so `pricer` imports work).

**Baseline (Day 5):** ~82–96 error with 100 train / 50 val, simple prompt, 1 epoch, batch_size=1.

**Optimizations:**
1. **Richer prompt** – Include category and Amazon US context
2. **More training data** – 200 train / 50 val (more signal without heavy overfitting)
3. **System prompt** – Clear instruction for consistent output format
4. **Hyperparameters** – `batch_size=4`, `learning_rate_multiplier=0.5`
5. **Stratified sampling** – Balance price bands and categories in train/val

In [2]:
import os
import sys

# Add week6 to path so pricer can be imported (works from repo root, week6, or community-contributions)
for _p in [".", os.path.join(os.getcwd(), ".."), os.path.join(os.getcwd(), "week6")]:
    _p = os.path.abspath(_p)
    if os.path.isdir(os.path.join(_p, "pricer")):
        sys.path.insert(0, _p)
        break

import re
import json
import random
from collections import defaultdict
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)
openai = OpenAI()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
LITE_MODE = False
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

README.md:   0%|          | 0.00/744 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/243M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/3.04M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/3.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 800,000 train, 10,000 val, 10,000 test


## Optimization 1: Stratified sampling

Ensure train/val include a balanced mix of price bands and categories.

In [17]:
def price_band(p):
    if p < 50: return "low"
    if p < 150: return "mid"
    return "high"

def stratified_sample(items, n_train=200, n_val=50, seed=42):
    random.seed(seed)
    by_strata = defaultdict(list)
    for item in items:
        key = (item.category, price_band(item.price))
        by_strata[key].append(item)
    train_out, val_out = [], []
    ratio = n_train / (n_train + n_val)
    for stratum, pool in by_strata.items():
        random.shuffle(pool)
        n = len(pool)
        split = max(0, min(n - 1, int(n * ratio)))
        train_out.extend(pool[:split])
        val_out.extend(pool[split:])
    random.shuffle(train_out)
    random.shuffle(val_out)
    fine_tune_train = train_out[:n_train]
    fine_tune_val = val_out[:n_val] if len(val_out) >= n_val else val_out
    return fine_tune_train, fine_tune_val

fine_tune_train, fine_tune_val = stratified_sample(train + val, n_train=200, n_val=50)
print(f"Optimized split: {len(fine_tune_train)} train, {len(fine_tune_val)} val")

Optimized split: 200 train, 50 val


## Optimization 2 & 3: Richer prompt + system message

Include category and Amazon US context; add a system prompt for consistent output format.

In [18]:
SYSTEM_PROMPT = "You are a product pricing expert. Estimate the price in USD. Respond with only the price in format $X.XX, no other text."

def messages_for_optimized(item):
    user_msg = (
        f"Category: {item.category}\n\n"
        f"Product: {item.summary}\n\n"
        f"Estimate the price in USD. Use Amazon US 2023 as reference. Respond with the price only, no explanation."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

def test_messages_for_optimized(item):
    user_msg = (
        f"Category: {item.category}\n\n"
        f"Product: {item.summary}\n\n"
        f"Estimate the price in USD. Use Amazon US 2023 as reference. Respond with the price only, no explanation."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg}
    ]

## Create JSONL and upload

In [19]:
def make_jsonl_optimized(items):
    lines = []
    for item in items:
        messages = messages_for_optimized(item)
        lines.append(json.dumps({"messages": messages}))
    return "\n".join(lines)

os.makedirs("jsonl", exist_ok=True)

with open("jsonl/optimized_train.jsonl", "w") as f:
    f.write(make_jsonl_optimized(fine_tune_train))
with open("jsonl/optimized_validation.jsonl", "w") as f:
    f.write(make_jsonl_optimized(fine_tune_val))

print(f"Wrote jsonl/optimized_train.jsonl ({len(fine_tune_train)} examples)")
print(f"Wrote jsonl/optimized_validation.jsonl ({len(fine_tune_val)} examples)")

Wrote jsonl/optimized_train.jsonl (200 examples)
Wrote jsonl/optimized_validation.jsonl (50 examples)


In [20]:
with open("jsonl/optimized_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
with open("jsonl/optimized_validation.jsonl", "rb") as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")
print(f"Uploaded: {train_file.id}, {val_file.id}")

Uploaded: file-UmfdSfiQB2rGmAqboEVW8t, file-WwdD8GhXyN4Tospbwjy4ed


## Optimization 4: Fine-tune with tuned hyperparameters

`batch_size=4`, `learning_rate_multiplier=0.5`, `n_epochs=1`

In [ ]:
job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={
        "n_epochs": 1,
        "batch_size": 4,
        "learning_rate_multiplier": 0.5,
    },
    suffix="pricer-optimized",
)
print(f"Job: {job.id}")

Job: ftjob-UvuVdyP1ZJuCy4qjJeocAqxC


## Wait for job completion

In [22]:
import time

job_id = job.id
while True:
    status = openai.fine_tuning.jobs.retrieve(job_id)
    print(status.status)
    if status.status == "succeeded":
        fine_tuned_model_name = status.fine_tuned_model
        print(f"Model: {fine_tuned_model_name}")
        break
    elif status.status == "failed":
        print(status)
        break
    time.sleep(30)

validating_files
validating_files
validating_files
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
running
succeeded
Model: ft:gpt-4.1-nano-2025-04-14:personal:pricer-optimized:DGMBjGlc


## Inference and evaluation

In [23]:
def optimized_pricer(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for_optimized(item),
        max_tokens=10,
        seed=42,
    )
    return response.choices[0].message.content

In [24]:
evaluate(optimized_pricer, test, size=200)

  0%|          | 0/200 [00:00<?, ?it/s]

$140 $146 $25 $40 $87 $70 $3 $119 $4 $135 $476 $51 $581 $41 $46 $13 $19 $36 $29 $0 $99 $46 $10 $37 $123 $224 $395 $796 $62 $30 $9 $20 $103 $736 $65 $420 $60 $16 $69 $9 $173 $62 $781 $531 $61 $2 $143 $18 $117 $21 $13 $103 $224 $20 $72 $40 $9 $222 $492 $33 $191 $58 $57 $15 $462 $139 $67 $325 $56 $41 $28 $4 $105 $27 $29 $22 $57 $4 $13 $10 $22 $15 $10 $44 $98 $107 $206 $104 $16 $54 $33 $15 $796 $21 $16 $9 $8 $32 $8 $220 $25 $3 $7 $67 $6 $112 $23 $275 $4 $44 $40 $120 $22 $46 $26 $860 $12 $1 $174 $22 $0 $341 $89 $83 $75 $99 $90 $140 $43 $31 $203 $44 $124 $8 $186 $1 $57 $60 $38 $22 $36 $449 $59 $16 $25 $148 $29 $271 $265 $9 $51 $234 $23 $40 $11 $507 $192 $13 $90 $33 $534 $18 $9 $14 $9 $3 $448 $33 $2 $70 $2 $5 $80 $144 $71 $299 $12 $45 $30 $77 $25 $5 $51 $151 $111 $18 $62 $163 $40 $23 $48 $190 $45 $19 $67 $40 $92 $42 $21 $10 

## Baseline comparison (optional)

Run this to compare with the Day 5 baseline style (100 train, simple prompt). If you have a baseline fine-tuned model ID, set it below.

In [26]:
def messages_baseline(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": msg}, {"role": "assistant", "content": f"${item.price:.2f}"}]

def test_messages_baseline(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": msg}]

# Uncomment and set your baseline model to compare:
BASELINE_MODEL = "ft:gpt-4.1-nano-..."
def baseline_pricer(item):
    r = openai.chat.completions.create(model=BASELINE_MODEL, messages=test_messages_baseline(item), max_tokens=7)
    return r.choices[0].message.content
evaluate(baseline_pricer, test, size=200)

  0%|          | 0/200 [00:00<?, ?it/s]

NotFoundError: Error code: 404 - {'error': {'message': 'The model `ft:gpt-4.1-nano-...` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}